# Create Swedish Energy Agency Awards

Creates Swedish Energy Agency (Energimyndigheten) awards from **SweCRIS** (Sweden's national research-information system, Vetenskapsrådet).

**Prerequisites:** run `scripts/local/swecris_to_s3.py --skip-upload` first (one reusable adapter; pulls the public SweCRIS REST API `swecris-api.vr.se/v1/scp/search`, splits all 68,843 Swedish funding decisions by `fundingOrganisationId` into the un-ingested target funders).

**S3:** `s3a://openalex-ingest/awards/swedish_energy_agency/swedish_energy_agency_projects.parquet` · **OpenAlex funder:** Swedish Energy Agency (Energimyndigheten) · `funder_id 4320322711` · SE.

**Schema notes (rich):**
- `funder_award_id` = SweCRIS `projectId` — the grant reference cited in papers (100%).
- `amount` = SEK (`fundingsSek`); 0/neg -> NULL.
- `lead_investigator` = `peopleList[0]` (given/family + **ORCID**) + `coordinatingOrganisation` (recipient institution), country Sweden.
- `display_name` = English title (Swedish fallback); `description` = abstract.

**Provenance** `swecris_energimyndigheten`, **priority 219** (Claude = odd). Same reusable SweCRIS adapter (Bearer token = the web app's own public token). CHECK-FIRST: VR/Vinnova/Formas/Forte/RJ already ingested — skipped.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.swedish_energy_agency_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/swedish_energy_agency/swedish_energy_agency_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.swedish_energy_agency_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.swedish_energy_agency_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.swedish_energy_agency_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast
Must return exactly 1 row. If 0, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi FROM openalex.common.funder WHERE funder_id = 4320322711;

## Step 2: Create Swedish Energy Agency Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.swedish_energy_agency_awards
USING delta
AS
WITH
fdr AS (SELECT funder_id, display_name, ror_id, doi FROM openalex.common.funder WHERE funder_id = 4320322711),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'SEK' ELSE NULL END as currency,
        struct(CONCAT('https://openalex.org/F', f.funder_id) as id, f.display_name, f.ror_id, f.doi) as funder,
        'research' as funding_type,
        g.funding_type_raw as funder_scheme,
        'swecris_energimyndigheten' as provenance,
        CASE WHEN TRY_CAST(g.start_year AS INT) IS NOT NULL THEN CAST(CONCAT(g.start_year,'-01-01') AS DATE) ELSE NULL END as start_date,
        CASE WHEN TRY_CAST(g.end_year AS INT) IS NOT NULL THEN CAST(CONCAT(g.end_year,'-12-31') AS DATE) ELSE NULL END as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        TRY_CAST(g.end_year AS INT) as end_year,
        CASE WHEN g.pi_family_name IS NOT NULL OR g.institution_name IS NOT NULL THEN
            struct(
                g.pi_given_name as given_name,
                g.pi_family_name as family_name,
                g.pi_orcid as orcid,
                struct(
                    g.institution_name as name,
                    'Sweden' as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation,
                CAST(NULL AS DATE) as role_start
            )
        ELSE NULL END as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>, role_start:DATE>>) as investigators,
        'https://www.swecris.se/' as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.swedish_energy_agency_raw g
    CROSS JOIN fdr f
    WHERE g.funder_award_id IS NOT NULL AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw WHERE provenance = 'swecris_energimyndigheten' AND priority = 219;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date, 219 as priority
FROM openalex.awards.swedish_energy_agency_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.swedish_energy_agency_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.orcid, lead_investigator.affiliation.name
FROM openalex.awards.swedish_energy_agency_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a freq check. n>5 combos should be real prolific Swedish PIs (verify title/institution diversity; ORCID disambiguates). Not a scraper bug — clean JSON API.
SELECT lead_investigator.given_name AS g, lead_investigator.family_name AS f, COUNT(*) AS n
FROM openalex.awards.swedish_energy_agency_awards GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(amount) has_amount, COUNT(lead_investigator.family_name) has_pi,
    COUNT(lead_investigator.orcid) has_orcid, COUNT(lead_investigator.affiliation.name) has_inst,
    ROUND(try_divide(COUNT(amount)*100.0, COUNT(*)),1) pct_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name)*100.0, COUNT(*)),1) pct_inst
FROM openalex.awards.swedish_energy_agency_awards;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name inst, COUNT(*) n
FROM openalex.awards.swedish_energy_agency_awards WHERE lead_investigator.affiliation.name IS NOT NULL GROUP BY 1 ORDER BY n DESC LIMIT 15;